# Sentinel-1 before/after imagery for the 2026 Nepal landslide

Search the Copernicus Data Space STAC catalog for Sentinel-1 SLC acquisitions around the 26 August 2026 Nepal–Tibet border landslide. The selection keeps before/after scenes on the same relative orbit and pass direction so they are suitable for comparison.

The event date in the GeoJSON has day precision. Acquisitions on that date are therefore shown separately and are not automatically treated as before or after. If no unambiguous post-event acquisition has been published yet, the notebook selects the closest pre-event scenes and can simply be rerun later.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import logging

import geopandas as gpd
import pandas as pd

from eo_tools.S1.download import download_partial_products, search_products
from eo_tools.util import explore_products
from eo_tools_dev.util import serve_map

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

## Configuration

In [ ]:
aoi_file = Path("../data/nepal_landslide_bounding_box.geojson")
data_dir = Path("/data/S1/partial_dls/Nepal_landslide_2026")
output_root = Path("/data/res/nepal-landslide-s1")
credentials_file = Path("/data/creds_s3.json")

days_before = 45
days_after = 30
scenes_per_period = 2

output_root.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

## Read the event footprint and date

In [ ]:
event = gpd.read_file(aoi_file).to_crs("EPSG:4326")
if len(event) != 1 or event.geometry.iloc[0].geom_type != "Polygon":
    raise ValueError("Expected exactly one Polygon feature in the event GeoJSON")

shp = event.geometry.iloc[0]
event_day = pd.Timestamp(event.iloc[0]["date"])
search_start = event_day - pd.Timedelta(days=days_before)
search_end = event_day + pd.Timedelta(days=days_after)

print(event.iloc[0]["name"])
print(f"Event date: {event_day.date()}")
print(f"Search window: {search_start.date()} to {search_end.date()}")
event.explore(tooltip=["name", "date", "description"] )

## Search Sentinel-1 SLC products

In [ ]:
products = search_products(
    intersects=shp,
    datetime=[search_start.strftime("%Y-%m-%d"), search_end.strftime("%Y-%m-%d")],
)

if products.empty:
    raise RuntimeError("No Sentinel-1 SLC products found in the search window")

products = products.set_crs("EPSG:4326", allow_override=True).copy()
products["acquired"] = pd.to_datetime(
    products["startTimeFromAscendingNode"], utc=True
).dt.tz_localize(None)

event_day_end = event_day + pd.Timedelta(days=1)
products["period"] = "event-day (review manually)"
products.loc[products.acquired < event_day, "period"] = "before"
products.loc[products.acquired >= event_day_end, "period"] = "after"
products = products.sort_values("acquired").reset_index(drop=True)

display(products[["acquired", "period", "relativeOrbitNumber", "orbitDirection", "id"]])
display(
    products.groupby(
        ["relativeOrbitNumber", "orbitDirection", "period"]
    ).size().rename("scene_count").to_frame()
)
products.drop(columns="stac_item").to_file(
    output_root / "products.geojson", driver="GeoJSON"
)

In [ ]:
m = explore_products(products=products, aoi=shp)
serve_map(m)

## Select a comparable orbit and the closest scenes

Prefer an orbit/pass with acquisitions on both sides of the event. Until one exists, use the orbit with the closest pre-event coverage. Event-day scenes remain in the table above for manual review but are excluded from automatic selection because the event time is unknown.

In [ ]:
unambiguous = products[products.period.isin(["before", "after"])].copy()
if unambiguous.empty:
    raise RuntimeError("Only event-day products were found; review them manually")

orbit_cols = ["relativeOrbitNumber", "orbitDirection"]
orbit_options = []
for orbit_key, group in unambiguous.groupby(orbit_cols):
    before = group[group.period == "before"]
    after = group[group.period == "after"]
    if before.empty:
        continue
    orbit_options.append(
        {
            "orbit_key": orbit_key,
            "has_after": not after.empty,
            "before_gap": (event_day - before.acquired.max()).total_seconds(),
            "after_gap": (after.acquired.min() - event_day_end).total_seconds() if not after.empty else float("inf"),
            "count": len(group),
        }
    )

if not orbit_options:
    raise RuntimeError("No pre-event Sentinel-1 acquisition was found")

best = sorted(
    orbit_options,
    key=lambda row: (not row["has_after"], row["before_gap"] + row["after_gap"], row["before_gap"], -row["count"]),
)[0]
relative_orbit, pass_direction = best["orbit_key"]
same_orbit = unambiguous[
    (unambiguous.relativeOrbitNumber == relative_orbit)
    & (unambiguous.orbitDirection == pass_direction)
].copy()

selected_before = same_orbit[same_orbit.period == "before"].nlargest(
    scenes_per_period, "acquired"
)
selected_after = same_orbit[same_orbit.period == "after"].nsmallest(
    scenes_per_period, "acquired"
)
sel = pd.concat([selected_before, selected_after]).sort_values("acquired")
sel = gpd.GeoDataFrame(sel, geometry="geometry", crs=products.crs)

print(f"Selected relative orbit {relative_orbit} ({pass_direction})")
if selected_after.empty:
    print("No post-event acquisition is available on this orbit yet. Rerun the search later.")
display(sel[["acquired", "period", "relativeOrbitNumber", "orbitDirection", "id"]])

sel.drop(columns="stac_item").to_file(
    output_root / "selected_products.geojson", driver="GeoJSON"
)
sel.drop(columns=["stac_item", "geometry"]).to_csv(
    output_root / "selected_products.csv", index=False
)

In [ ]:
m = explore_products(products=sel, aoi=shp)
serve_map(m)

## Download partial Sentinel-1 products

Set `RUN_DOWNLOAD` to `True` after checking the selected footprints. CDSE S3 credentials are expected in `/data/creds_s3.json`, matching the Berlin notebook. If there is no post-event scene yet, this downloads only the selected pre-event products; rerunning the search and download cells later will add post-event products without overwriting existing downloads.

In [ ]:
RUN_DOWNLOAD = False

if RUN_DOWNLOAD:
    if not credentials_file.exists():
        raise FileNotFoundError(f"CDSE S3 credentials not found: {credentials_file}")
    with credentials_file.open() as f:
        cred = json.load(f)

    download_partial_products(
        sel,
        shp,
        out_dir=data_dir,
        aws_key=cred["username"],
        aws_secret=cred["password"],
        pol="full",
        force_overwrite=False,
    )
else:
    print("Dry run: set RUN_DOWNLOAD = True to download the selected partial products.")

In [ ]:
partial_products = sorted(data_dir.glob("*.partial.SAFE"))
print(f"Found {len(partial_products)} downloaded partial products in {data_dir}")
for product_dir in partial_products:
    print(product_dir.name)